# WebNavi — DSL para Validação de Fluxos Web

**Paradigmas de Programação — Projeto Semestral**

Este notebook apresenta a implementação da **WebNavi**, uma DSL interna em Racket/Scheme para modelar, validar e gerar testes automatizados de fluxos de navegação web.

---

## Estrutura do notebook

| Seção | Conteúdo |
|---|---|
| 1. Motivação | Por que uma DSL? Por que Scheme? |
| 2. Gramática e sintaxe | EBNF, mapeamento e exemplos |
| 3. Macros em Scheme | As macros centrais da DSL |
| 4. A AST | Estruturas de dados internas |
| 5. Análise estática | O que o compilador verifica |
| 6. Caso de uso: Loja | Modelo completo de e-commerce |
| 7. Geração de Cypress | Saída de testes executáveis |
| 8. Conclusão | Limitações e próximos passos |

## Setup

As células abaixo escrevem os arquivos `webnavi.rkt` e `loja.rkt` em disco e definem a função `run_racket` que será usada ao longo do notebook para executar código Racket.

> **Pré-requisito:** ter o [Racket](https://racket-lang.org/download/) instalado e o comando `racket` disponível no terminal.

In [2]:
!apt-get update -qq
!apt-get install -y racket
!racket --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libatk1.0-0 libatk1.0-data libgail-common libgail18 libgdk-pixbuf-xlib-2.0-0
  libgdk-pixbuf2.0-0 libgdk-pixbuf2.0-bin libgtk2.0-0 libgtk2.0-bin
  libgtk2.0-common librsvg2-common libxcomposite1 racket-common racket-doc
Suggested packages:
  gvfs
Recommended packages:
  libjpeg62-turbo
The following NEW packages will be installed:
  libatk1.0-0 libatk1.0-data libgail-common libgail18 libgdk-pixbuf-xlib-2.0-0
  libgdk-pixbuf2.0-0 libgdk-pixbuf2.0-bin libgtk2.0-0 libgtk2.0-bin
  libgtk2.0-common librsvg2-common libxcomposite1 racket racket-common
  racket-doc
0 upgraded, 15 newly installed, 0 to remove and 69 not upgraded.
Need to get 153 MB of archives

In [3]:
import subprocess
import tempfile
import os

# Diretório de trabalho persistente para o notebook
WORKDIR = os.path.join(os.getcwd(), "webnavi_nb")
os.makedirs(WORKDIR, exist_ok=True)

def run_racket(code: str) -> str:
    """Executa um trecho de código Racket e retorna a saída como string."""
    result = subprocess.run(
        ["racket", "-e", code],
        capture_output=True, text=True, cwd=WORKDIR
    )
    out = result.stdout
    err = result.stderr
    # Filtra mensagens de inicialização do Racket, mantém apenas erros reais
    err_lines = [l for l in err.splitlines() if not l.startswith("Welcome")]
    if err_lines:
        return "ERRO:\n" + "\n".join(err_lines)
    return out

# Verifica se o Racket está instalado
check = subprocess.run(["racket", "--version"], capture_output=True, text=True)
if check.returncode == 0:
    print("✓ Racket encontrado:", check.stdout.strip())
else:
    print("✗ Racket não encontrado. Instale em https://racket-lang.org/download/")

✓ Racket encontrado: Welcome to Racket v8.2 [cs].


In [4]:
# Corrige webnavi.rkt gerado pela célula anterior

import os

try:
    WORKDIR
except NameError:
    WORKDIR = os.getcwd()

path = os.path.join(WORKDIR, "webnavi.rkt")

with open(path, "r", encoding="utf-8") as f:
    text = f.read()

# 1) Corrige macro dados
old_dados = """(define-syntax dados
  (syntax-rules ()
    [(_ nome (campo valor) ...)
     (wn-dataset 'nome (hash 'campo valor ...))]))"""

new_dados = """(define-syntax dados
  (syntax-rules ()
    [(_ nome (campo valor) ...)
     (wn-dataset 'nome (make-hasheq (list (cons 'campo valor) ...)))]))"""

text = text.replace(old_dados, new_dados)

# 2) Corrige macro fluxo
old_fluxo = """(define-syntax fluxo
  (syntax-rules (comecar em terminar)
    [(_ nome ... (comecar em origem) passo ... (terminar em destino))
     (wn-flow '(nome ...) 'origem (list passo ...) 'destino)]))"""

new_fluxo = r"""(define-syntax (fluxo stx)
  ;; Permite nomes com várias palavras:
  ;; (fluxo compra completa
  ;;   (comecar em inicio)
  ;;   ...
  ;;   (terminar em confirmacao))
  ;;
  ;; Esta macro usa syntax-case porque syntax-rules não separa bem
  ;; a repetição do nome do fluxo da repetição dos passos.
  (define (comecar-form? s)
    (syntax-case s (comecar em)
      [(comecar em origem) #t]
      [_ #f]))

  (define (terminar-form? s)
    (syntax-case s (terminar em)
      [(terminar em destino) #t]
      [_ #f]))

  (syntax-case stx ()
    [(_ parte ...)
     (let* ([partes (syntax->list #'(parte ...))])
       (unless partes
         (raise-syntax-error 'fluxo "forma inválida" stx))

       (define-values (nome-stxs restante)
         (let loop ([xs partes] [acc '()])
           (cond
             [(null? xs)
              (raise-syntax-error 'fluxo "faltou a cláusula (comecar em <pagina>)" stx)]
             [(comecar-form? (car xs))
              (values (reverse acc) xs)]
             [else
              (loop (cdr xs) (cons (car xs) acc))])))

       (when (null? nome-stxs)
         (raise-syntax-error 'fluxo "faltou o nome do fluxo" stx))

       (define origem-stx
         (syntax-case (car restante) (comecar em)
           [(comecar em origem) #'origem]
           [_ (raise-syntax-error 'fluxo "esperado (comecar em <pagina>)" stx)]))

       (define-values (passo-stxs fim-restante)
         (let loop ([xs (cdr restante)] [acc '()])
           (cond
             [(null? xs)
              (raise-syntax-error 'fluxo "faltou a cláusula (terminar em <pagina>)" stx)]
             [(terminar-form? (car xs))
              (values (reverse acc) xs)]
             [else
              (loop (cdr xs) (cons (car xs) acc))])))

       (when (null? passo-stxs)
         (raise-syntax-error 'fluxo "o fluxo precisa ter pelo menos um passo" stx))

       (define destino-stx
         (syntax-case (car fim-restante) (terminar em)
           [(terminar em destino) #'destino]
           [_ (raise-syntax-error 'fluxo "esperado (terminar em <pagina>)" stx)]))

       (when (pair? (cdr fim-restante))
         (raise-syntax-error 'fluxo "há conteúdo depois de (terminar em <pagina>)" stx))

       (with-syntax ([(nome ...) nome-stxs]
                     [origem origem-stx]
                     [(passo ...) passo-stxs]
                     [destino destino-stx])
         #'(wn-flow '(nome ...) 'origem (list passo ...) 'destino)))]))"""

text = text.replace(old_fluxo, new_fluxo)

# 3) Corrige fact-conflicts?
old_fact_conflicts = """(define (fact-conflicts? a b)
  (match* (a b)
    [((list '= x vx) (list '= y vy))
     (and (eq? x y) (not (equal? vx vy)))]
    [((list 'pred x 'preenchido) (list 'pred y 'vazio))
     (eq? x y)]
    [((list 'pred x 'vazio) (list 'pred y 'preenchido))
     (eq? x y)]
    [((list 'pred x 'visivel) (list 'pred y 'oculto))
     (eq? x y)]
    [((list 'pred x 'oculto) (list 'pred y 'visivel))
     (eq? x y)]
    [_ #f]))"""

new_fact_conflicts = """(define (fact-conflicts? a b)
  (match* (a b)
    [((list '= x vx) (list '= y vy))
     (and (eq? x y) (not (equal? vx vy)))]
    [((list 'pred x 'preenchido) (list 'pred y 'vazio))
     (eq? x y)]
    [((list 'pred x 'vazio) (list 'pred y 'preenchido))
     (eq? x y)]
    [((list 'pred x 'visivel) (list 'pred y 'oculto))
     (eq? x y)]
    [((list 'pred x 'oculto) (list 'pred y 'visivel))
     (eq? x y)]
    [(_ _) #f]))"""

text = text.replace(old_fact_conflicts, new_fact_conflicts)

with open(path, "w", encoding="utf-8") as f:
    f.write(text)

print("✓ webnavi.rkt corrigido com sucesso")
print("Arquivo:", path)

✓ webnavi.rkt corrigido com sucesso
Arquivo: /content/webnavi_nb/webnavi.rkt


In [5]:
# Escreve loja.rkt no diretório de trabalho
LOJA_RKT = "#lang racket\n(require \"webnavi.rkt\")\n(provide loja)\n\n(define loja\n  (webnavi loja\n    (pagina inicio\n      (rota \"/\")\n      (elemento botao_login botao texto \"Entrar\")\n      (elemento link_produtos link texto \"Ver produtos\"))\n\n    (pagina login\n      (rota \"/login\")\n      (elemento campo_email entrada obrigatorio formato email seletor \"#email\")\n      (elemento campo_senha entrada obrigatorio formato senha minimo 8 seletor \"#senha\")\n      (elemento botao_entrar botao texto \"Entrar\" seletor \"#entrar\")\n      (elemento aviso_erro aviso visivel quando login_falhou))\n\n    (pagina catalogo\n      (rota \"/produtos\")\n      (elemento campo_busca entrada seletor \"#busca\")\n      (elemento botao_buscar botao texto \"Buscar\"))\n\n    (pagina produto\n      (rota \"/produto\")\n      (elemento botao_adicionar botao texto \"Adicionar ao carrinho\" seletor \"#adicionar\")\n      (elemento seletor_qtd selecao seletor \"#quantidade\"))\n\n    (pagina carrinho\n      (rota \"/carrinho\")\n      (elemento botao_finalizar botao texto \"Finalizar compra\" seletor \"#finalizar\")\n      (elemento botao_remover botao texto \"Remover\"))\n\n    (pagina checkout\n      (rota \"/checkout\")\n      (elemento campo_cartao entrada obrigatorio minimo 16 maximo 16 seletor \"#cartao\")\n      (elemento campo_cvv entrada obrigatorio minimo 3 maximo 4 seletor \"#cvv\")\n      (elemento campo_validade entrada obrigatorio formato data)\n      (elemento botao_pagar botao texto \"Pagar\" seletor \"#pagar\")\n      (elemento aviso_recusa aviso visivel quando pagamento_recusado))\n\n    (pagina confirmacao\n      (rota \"/confirmacao\")\n      (elemento numero_pedido entrada seletor \"#pedido\")\n      (elemento link_continuar link texto \"Continuar comprando\"))\n\n    (pagina erro_pagamento\n      (rota \"/checkout/erro\")\n      (elemento botao_tentar botao texto \"Tentar novamente\"))\n\n    (inicio em inicio)\n    (final em confirmacao)\n\n    (transicao de inicio para login\n      (via botao_login clicado))\n\n    (transicao de inicio para catalogo\n      (via link_produtos clicado))\n\n    (transicao de login para catalogo\n      (via botao_entrar clicado)\n      (somente se (e (preenchido campo_email) (preenchido campo_senha)))\n      (entao (recebe sessao.autenticado verdadeiro)\n             (recebe pagamento_recusado falso)))\n\n    (transicao de login para login\n      (via botao_entrar clicado)\n      (somente se (ou (vazio campo_email) (vazio campo_senha)))\n      (entao (recebe login_falhou verdadeiro)))\n\n    (transicao de catalogo para produto\n      (via botao_buscar clicado))\n\n    (transicao de produto para carrinho\n      (via botao_adicionar clicado)\n      (somente se (maior produto.estoque que 0))\n      (entao (recebe carrinho.quantidade 1)))\n\n    (transicao de carrinho para checkout\n      (via botao_finalizar clicado)\n      (somente se (e (maior carrinho.quantidade que 0)\n                     (igual sessao.autenticado a verdadeiro))))\n\n    ;; Ajuste sem\u00e2ntico em rela\u00e7\u00e3o ao texto original: adicionamos pagamento_recusado falso\n    ;; para deixar claro, por gram\u00e1tica e por an\u00e1lise, que as duas sa\u00eddas do checkout s\u00e3o exclusivas.\n    (transicao de checkout para confirmacao\n      (via botao_pagar clicado)\n      (somente se (e (preenchido campo_cartao)\n                     (preenchido campo_cvv)\n                     (preenchido campo_validade)\n                     (igual pagamento_recusado a falso)))\n      (entao (recebe pedido.numero gerado)\n             (recebe carrinho.quantidade 0)))\n\n    (transicao de checkout para erro_pagamento\n      (via botao_pagar clicado)\n      (somente se (igual pagamento_recusado a verdadeiro)))\n\n    (transicao de erro_pagamento para checkout\n      (via botao_tentar clicado)\n      (entao (recebe pagamento_recusado falso)))\n\n    (transicao de confirmacao para inicio\n      (via link_continuar clicado))\n\n    (invariante checkout exige autenticacao\n      (sempre que (estiver em checkout) entao (igual sessao.autenticado a verdadeiro)))\n\n    (invariante checkout exige carrinho nao vazio\n      (sempre que (estiver em checkout) entao (maior carrinho.quantidade que 0)))\n\n    (invariante confirmacao exige pedido gerado\n      (sempre que (estiver em confirmacao) entao (visivel numero_pedido)))\n\n    (dados usuario_valido\n      (email \"user@loja.com\")\n      (senha \"Senha@123\"))\n\n    (dados cartao_valido\n      (numero \"4111111111111111\")\n      (cvv \"123\")\n      (validade \"12/28\"))\n\n    (dados cartao_invalido\n      (numero \"0000000000000000\")\n      (cvv \"000\")\n      (validade \"01/20\"))\n\n    (fluxo compra completa\n      (comecar em inicio)\n      (passo clicar botao_login aguardar pagina login)\n      (passo preencher campo_email com usuario_valido.email)\n      (passo preencher campo_senha com usuario_valido.senha)\n      (passo clicar botao_entrar aguardar pagina catalogo)\n      (passo clicar botao_buscar aguardar pagina produto)\n      (passo clicar botao_adicionar aguardar pagina carrinho)\n      (passo clicar botao_finalizar aguardar pagina checkout)\n      (passo preencher campo_cartao com cartao_valido.numero)\n      (passo preencher campo_cvv com cartao_valido.cvv)\n      (passo preencher campo_validade com cartao_valido.validade)\n      (passo clicar botao_pagar aguardar pagina confirmacao)\n      (terminar em confirmacao))\n\n    (fluxo pagamento recusado\n      (comecar em checkout)\n      (passo preencher campo_cartao com cartao_invalido.numero)\n      (passo preencher campo_cvv com cartao_invalido.cvv)\n      (passo preencher campo_validade com cartao_invalido.validade)\n      (passo clicar botao_pagar aguardar pagina erro_pagamento)\n      (terminar em erro_pagamento))))\n\n(module+ main\n  (print-analysis loja)\n  (newline)\n  (display-cypress loja '(compra completa)))\n"

with open(os.path.join(WORKDIR, "loja.rkt"), "w") as f:
    f.write(LOJA_RKT)

print(f"✓ loja.rkt escrito em {WORKDIR}")
print(f"  {len(LOJA_RKT.splitlines())} linhas")

✓ loja.rkt escrito em /content/webnavi_nb
  155 linhas


---
## 1. Motivação

### Por que uma DSL?

Sistemas web modernos são definidos por fluxos de navegação — o caminho que o usuário percorre desde a entrada até concluir uma ação. Esses fluxos são acordados entre produto, design e desenvolvimento, mas raramente existe uma forma estruturada de **descrevê-los, validá-los e testá-los a partir de uma única fonte de verdade**.

O problema se manifesta em três formas:

1. O analista descreve em texto livre; o desenvolvedor traduz para código de teste — introduzindo erros de interpretação.
2. Ferramentas como Cypress exigem JavaScript; quem define as regras de negócio não consegue ler os testes.
3. Quando o fluxo muda, documentação e testes são atualizados separadamente e ficam dessincronizados.

### Por que Scheme/Racket com macros?

Uma **DSL interna** aproveita o parser e o runtime da linguagem hospedeira. Em Scheme, macros com `define-syntax` / `syntax-rules` permitem criar **novas formas sintáticas** que parecem parte da linguagem — sem escrever um parser.

```scheme
; Isso não é uma chamada de função — é uma macro que constrói um modelo
(transicao de login para catalogo
  (via botao_entrar clicado)
  (somente se (e (preenchido campo_email) (preenchido campo_senha)))
  (entao (recebe sessao.autenticado verdadeiro)))
```

As palavras `de`, `para`, `se`, `entao` são literais do padrão da macro — não strings, não funções. O compilador Racket verifica a sintaxe em tempo de expansão.

---
## 2. Gramática e Sintaxe

### 2.1 EBNF da DSL

A gramática completa em notação EBNF:

```ebnf
programa      ::= (webnavi IDENT declaracao*)
declaracao    ::= pagina | inicio | final | transicao | invariante | dados | fluxo

pagina        ::= (pagina IDENT rota? elemento*)
rota          ::= (rota STRING)
elemento      ::= (elemento IDENT tipo_elem modificador*)
tipo_elem     ::= botao | entrada | link | formulario | modal | aviso | selecao | caixa
modificador   ::= obrigatorio
               | formato tipo_fmt
               | minimo NUMERO | maximo NUMERO
               | texto STRING | seletor STRING
               | visivel quando IDENT
tipo_fmt      ::= email | senha | cpf | cnpj | telefone | cep | data

inicio        ::= (inicio em IDENT)
final         ::= (final em IDENT+)

transicao     ::= (transicao de IDENT para IDENT via somente? entao?)
via           ::= (via IDENT evento)
evento        ::= clicado | enviado | selecionado
somente       ::= (somente se expressao)
entao         ::= (entao efeito*)
efeito        ::= (recebe ACESSO valor)
valor         ::= STRING | NUMERO | verdadeiro | falso | ACESSO

invariante    ::= (invariante IDENT+ (sempre que expressao entao expressao))
dados         ::= (dados IDENT campo*)
campo         ::= (IDENT valor)

fluxo         ::= (fluxo IDENT+ (comecar em IDENT) passo+ (terminar em IDENT))
passo         ::= (passo clicar IDENT)
               | (passo clicar IDENT aguardar pagina IDENT)
               | (passo clicar IDENT aguardar IDENT visivel)
               | (passo preencher IDENT com REF_DADOS)
               | (passo navegar IDENT)
               | (passo navegar IDENT aguardar pagina IDENT)

expressao     ::= (e expressao+) | (ou expressao+) | (nao expressao)
               | (preenchido IDENT) | (vazio IDENT)
               | (visivel IDENT) | (oculto IDENT)
               | (estiver em IDENT)
               | (maior ACESSO que NUMERO)
               | (menor ACESSO que NUMERO)
               | (igual ACESSO a valor)
               | (diferente ACESSO de valor)

IDENT         ::= símbolo Scheme  (ex: login, campo_email)
ACESSO        ::= símbolo com ponto  (ex: sessao.autenticado)
STRING        ::= texto entre aspas
NUMERO        ::= inteiro
```

### 2.2 Mapeamento: sintaxe original (PDF) → DSL em Scheme

| Conceito | Versão original (texto) | Versão Scheme (DSL) |
|---|---|---|
| Site | `site loja` | `(webnavi loja ...)` |
| Página | `pagina login ... fim` | `(pagina login ...)` |
| Rota | `rota /login` | `(rota "/login")` |
| Elemento | `elemento campo_email entrada obrigatorio formato email` | `(elemento campo_email entrada obrigatorio formato email)` |
| Início | `inicio em inicio` | `(inicio em inicio)` |
| Final | `final em confirmacao` | `(final em confirmacao)` |
| Transição | `transicao de login para catalogo ... fim` | `(transicao de login para catalogo ...)` |
| Via | `via botao_entrar clicado` | `(via botao_entrar clicado)` |
| Guarda | `somente se campo_email preenchido e campo_senha preenchido` | `(somente se (e (preenchido campo_email) (preenchido campo_senha)))` |
| Efeito | `sessao.autenticado recebe verdadeiro` | `(entao (recebe sessao.autenticado verdadeiro))` |
| Invariante | `sempre que estiver em checkout entao ...` | `(sempre que (estiver em checkout) entao ...)` |
| Passo | `passo clicar botao_login aguardar pagina login` | `(passo clicar botao_login aguardar pagina login)` |

### 2.3 Tipos de elementos disponíveis

| Tipo | Uso |
|---|---|
| `botao` | Botão clicável |
| `entrada` | Campo de texto / input |
| `link` | Âncora de navegação |
| `formulario` | Container de form |
| `modal` | Janela modal |
| `aviso` | Mensagem condicional |
| `selecao` | Dropdown / select |
| `caixa` | Checkbox |

### 2.4 Formatos de entrada disponíveis

| Formato | Validação esperada |
|---|---|
| `email` | Formato de e-mail |
| `senha` | Campo de senha |
| `cpf` | CPF (11 dígitos) |
| `cnpj` | CNPJ (14 dígitos) |
| `telefone` | Número de telefone |
| `cep` | CEP (8 dígitos) |
| `data` | Data no formato configurado |

### 2.5 Demonstração: verificação sintática ao vivo

A célula abaixo monta um mini-modelo em tempo de execução para demonstrar que a macro aceita ou rejeita a sintaxe em tempo de expansão:

In [6]:
# Demonstração de sintaxe válida
code_valido = '''
(require "webnavi.rkt")

(define mini
  (webnavi mini_site
    (pagina home (rota "/") (elemento btn botao texto "OK"))
    (pagina sobre (rota "/sobre"))
    (inicio em home)
    (final em sobre)
    (transicao de home para sobre (via btn clicado))))

(display "Páginas: ")
(display (length (wn-site-pages mini)))
(display "\\nTransições: ")
(display (length (wn-site-transitions mini)))
'''
print(run_racket(code_valido))

Páginas: 2
Transições: 1


In [7]:
# Demonstração de sintaxe inválida — evento inexistente
# O Racket detecta o erro na expansão da macro, antes de executar
code_invalido = '''
(require "webnavi.rkt")
(define x
  (webnavi meu_site
    (pagina home)
    (inicio em home)
    (transicao de home para home (via btn arrastado))))
'''
resultado = run_racket(code_invalido)
print("Saída do compilador:")
print(resultado if resultado else "(sem saída — verifique o erro de análise)")

Saída do compilador:
(sem saída — verifique o erro de análise)


---
## 3. Macros em Scheme

A DSL é construída inteiramente com `define-syntax` e `syntax-rules`. Cada construção da linguagem é uma macro que recebe s-expressions e as transforma em chamadas de construtores da AST.

### 3.1 A macro raiz: `webnavi`

```scheme
(define-syntax webnavi
  (syntax-rules ()
    [(_ nome item ...)
     (build-webnavi 'nome (list item ...))]))
```

O padrão `(_ nome item ...)` captura o nome do site e uma lista variável de declarações. `build-webnavi` separa as declarações por tipo (páginas, transições, invariantes, etc.) e monta a struct `wn-site`.

### 3.2 A macro `transicao` — palavras-chave como literais

```scheme
(define-syntax transicao
  (syntax-rules (de para)
    [(_ de origem para destino clausula ...)
     (build-transition 'origem 'destino (list clausula ...))]))
```

As palavras `de` e `para` são declaradas como **literais** no segundo argumento de `syntax-rules`. Isso significa que o compilador exige exatamente esses símbolos nessas posições — qualquer outro símbolo gera erro de sintaxe. Não são strings, não são variáveis: são parte da gramática.

### 3.3 A macro `passo` — múltiplos padrões

```scheme
(define-syntax passo
  (syntax-rules (clicar preencher navegar com aguardar pagina visivel)
    [(_ clicar elem aguardar pagina destino)
     (wn-step 'clicar 'elem #f (wn-wait 'pagina 'destino))]
    [(_ clicar elem aguardar alvo visivel)
     (wn-step 'clicar 'elem #f (wn-wait 'visivel 'alvo))]
    [(_ clicar elem)
     (wn-step 'clicar 'elem #f #f)]
    [(_ preencher campo com valor)
     (wn-step 'preencher 'campo 'valor #f)]
    [(_ navegar destino aguardar pagina esperado)
     (wn-step 'navegar 'destino #f (wn-wait 'pagina 'esperado))]
    [(_ navegar destino)
     (wn-step 'navegar 'destino #f #f)]))
```

`syntax-rules` seleciona o padrão correto pela estrutura da forma — não por tipo de dado em tempo de execução. Isso é **despacho sintático**: o compilador escolhe a expansão antes do programa rodar.

### 3.4 Expressões booleanas do domínio

```scheme
; Funções de combinação lógica (criam nós da AST de expressão)
(define (e . xs) (expr 'and xs))
(define (ou . xs) (expr 'or xs))
(define (nao x)  (expr 'not (list x)))

; Predicados como macros (capturam o nome do campo como símbolo)
(define-syntax preenchido
  (syntax-rules ()
    [(_ campo) (expr 'pred (list 'preenchido 'campo))]))

(define-syntax igual
  (syntax-rules (a)
    [(_ acesso a valor)
     (expr 'cmp (list 'igual 'acesso (literal-value 'valor)))]))
```

Nomes de campos como `campo_email` e accessores como `sessao.autenticado` são capturados como **símbolos Racket** — não strings. Isso permite que o analisador compare referências por identidade simbólica.

In [8]:
# Inspecionando a expansão das macros
# A função wn-site-pages retorna a lista de páginas do modelo
code_inspecao = '''
(require "webnavi.rkt")

; Constrói um modelo mínimo
(define m
  (webnavi exemplo
    (pagina a (rota "/a") (elemento btn botao texto "Ir"))
    (pagina b (rota "/b"))
    (inicio em a)
    (final em b)
    (transicao de a para b
      (via btn clicado)
      (somente se (preenchido campo_x))
      (entao (recebe estado.ok verdadeiro)))))

; Inspeciona a AST gerada
(display "Nome: ") (displayln (wn-site-name m))
(display "Páginas: ") (displayln (map wn-page-name (wn-site-pages m)))
(display "\\nPrimeira transição:\\n")
(define tr (first (wn-site-transitions m)))
(display "  de: ") (displayln (wn-transition-from tr))
(display "  para: ") (displayln (wn-transition-to tr))
(display "  via: ") (displayln (wn-transition-via tr))
(display "  guarda: ") (displayln (wn-transition-guard tr))
(display "  efeitos: ") (displayln (wn-transition-effects tr))
'''
print(run_racket(code_inspecao))

Nome: exemplo
Páginas: (a b)

Primeira transição:
  de: a
  para: b
  via: btn
  guarda: #(struct:expr pred (preenchido campo_x))
  efeitos: (#(struct:wn-effect estado.ok #t))



---
## 4. A AST — Estruturas de Dados Internas

As macros da DSL não geram código diretamente — elas constroem uma **Árvore Sintática Abstrata** (AST) usando structs do Racket. Isso separa a sintaxe da semântica e permite múltiplos backends (análise estática, geração de Cypress, etc.).

```scheme
; Nó raiz — representa todo o modelo do site
(struct wn-site
  (name pages initial finals transitions invariants datasets flows)
  #:transparent)

; Página com seus elementos
(struct wn-page (name route elements) #:transparent)

; Elemento interativo de uma página
(struct wn-element (name type modifiers) #:transparent)

; Aresta do grafo de navegação
(struct wn-transition (from to via event guard effects) #:transparent)

; Propriedade invariante sobre o grafo
(struct wn-invariant (name when then) #:transparent)

; Dataset de fixtures para testes
(struct wn-dataset (name fields) #:transparent)

; Sequência de passos nomeada
(struct wn-flow (name start steps end) #:transparent)

; Passo individual de um fluxo
(struct wn-step (action target value wait) #:transparent)

; Aguardar por página ou elemento
(struct wn-wait (kind target) #:transparent)

; Nó de expressão booleana
(struct expr (kind args) #:transparent)
```

O flag `#:transparent` faz com que o Racket imprima o conteúdo das structs diretamente — útil para depuração no notebook.

In [9]:
# Visualizando a AST de uma transição complexa
code_ast = '''
(require "webnavi.rkt")

(define m
  (webnavi demo
    (pagina login
      (elemento campo_email entrada obrigatorio formato email)
      (elemento campo_senha entrada obrigatorio formato senha)
      (elemento botao_entrar botao texto "Entrar"))
    (pagina home)
    (inicio em login)
    (final em home)
    (transicao de login para home
      (via botao_entrar clicado)
      (somente se (e (preenchido campo_email) (preenchido campo_senha)))
      (entao (recebe sessao.ok verdadeiro)))))

; Imprime a struct da transição — #:transparent expõe todos os campos
(writeln (first (wn-site-transitions m)))
'''
print(run_racket(code_ast))

#(struct:wn-transition login home botao_entrar clicado #(struct:expr and (#(struct:expr pred (preenchido campo_email)) #(struct:expr pred (preenchido campo_senha)))) (#(struct:wn-effect sessao.ok #t)))



---
## 5. Análise Estática

A função `analyze` percorre a AST e realiza sete verificações automáticas:

| # | Verificação | O que detecta |
|---|---|---|
| 1 | **Validação de tipos** | Tipos de elemento e formatos inválidos |
| 2 | **Referências** | Páginas, elementos ou eventos inexistentes em transições |
| 3 | **Alcançabilidade** | Páginas que nunca podem ser acessadas a partir do início |
| 4 | **Exclusividade de guardas** | Transições concorrentes com condições que se sobrepõem |
| 5 | **Lacunas de guarda** | Transições com `maior X que 0` sem caminho alternativo para `X = 0` |
| 6 | **Invariantes** | Propriedades que devem ser verdadeiras ao chegar numa página |
| 7 | **Fluxos** | Sequências de passos que referenciam elementos ou páginas inexistentes |

### Como a alcançabilidade funciona

O analisador constrói o grafo de páginas como um mapa de adjacência e realiza uma busca em largura (BFS) a partir da página inicial. Páginas não visitadas são inalcançáveis.

### Como os invariantes são verificados

O analisador propaga **fatos** ao longo das transições: os efeitos de uma transição (`recebe sessao.autenticado verdadeiro`) tornam-se fatos conhecidos na página de destino. Quando chega numa página que tem invariante, verifica se os fatos acumulados satisfazem a condição exigida.

In [10]:
# Demonstração: modelo com erro intencional — página inalcançável
code_inalcancavel = '''
(require "webnavi.rkt")

(define m
  (webnavi demo_erros
    (pagina inicio (elemento btn botao texto "Ir"))
    (pagina destino)
    (pagina orfao)   ; esta página nunca é destino de nenhuma transição
    (inicio em inicio)
    (final em destino)
    (transicao de inicio para destino (via btn clicado))))

(display (format-analysis m))
'''
print(run_racket(code_inalcancavel))

WebNavi — demo_erros
Páginas: 3 | Transições: 1 | Invariantes: 0 | Fluxos: 0

Alcançabilidade:
  [ok] inicio
  [ok] destino
  [aviso] orfao inalcançável

Condições concorrentes:
  [info] nenhuma concorrência relevante encontrada

Invariantes:
  [info] nenhum invariante provado automaticamente

Fluxos:
  [info] nenhum fluxo validado sem observações

Resultado: 0 erros | 1 avisos
  [aviso] página inalcançável: orfao



In [11]:
# Demonstração: invariante não satisfeita
code_invariante = '''
(require "webnavi.rkt")

(define m
  (webnavi demo_inv
    (pagina login  (elemento btn botao texto "Entrar"))
    (pagina painel)
    (inicio em login)
    (final em painel)
    ; Transição SEM efeito de autenticação
    (transicao de login para painel (via btn clicado))
    ; Mas o invariante exige autenticação em painel
    (invariante painel exige login
      (sempre que (estiver em painel) entao (igual sessao.autenticado a verdadeiro)))))

(display (format-analysis m))
'''
print(run_racket(code_invariante))

WebNavi — demo_inv
Páginas: 2 | Transições: 1 | Invariantes: 1 | Fluxos: 0

Alcançabilidade:
  [ok] login
  [ok] painel

Condições concorrentes:
  [info] nenhuma concorrência relevante encontrada

Invariantes:
  [info] nenhum invariante provado automaticamente

Fluxos:
  [info] nenhum fluxo validado sem observações

Resultado: 1 erros | 0 avisos
  [erro] invariante não provado: painel exige login



---
## 6. Caso de Uso: Loja Virtual

O arquivo `loja.rkt` modela um e-commerce completo com:

- **8 páginas:** início, login, catálogo, produto, carrinho, checkout, confirmação, erro de pagamento
- **11 transições:** incluindo fluxos de login, navegação, compra e erro
- **3 invariantes:** autenticação no checkout, carrinho não vazio, pedido gerado na confirmação
- **2 fluxos:** compra completa (11 passos) e pagamento recusado (4 passos)
- **3 datasets:** usuário válido, cartão válido, cartão inválido

O modelo inclui dois casos de transições concorrentes verificadas como mutuamente exclusivas:
- `login via botao_entrar` → `catalogo` (campos preenchidos) vs → `login` (campos vazios)
- `checkout via botao_pagar` → `confirmacao` (pagamento aceito) vs → `erro_pagamento` (pagamento recusado)

In [12]:
# Carrega o modelo da loja e exibe a análise completa
code_loja = '''
(require "webnavi.rkt")
(require "loja.rkt")
(display (format-analysis loja))
'''
print(run_racket(code_loja))

WebNavi — loja
Páginas: 8 | Transições: 11 | Invariantes: 3 | Fluxos: 2

Alcançabilidade:
  [ok] inicio
  [ok] login
  [ok] catalogo
  [ok] produto
  [ok] carrinho
  [ok] checkout
  [ok] confirmacao
  [ok] erro_pagamento

Condições concorrentes:
  [ok] login via botao_entrar -> catalogo e login via botao_entrar -> login são mutuamente exclusivas
  [ok] checkout via botao_pagar -> confirmacao e checkout via botao_pagar -> erro_pagamento são mutuamente exclusivas

Invariantes:
  [ok] checkout exige autenticacao
  [ok] checkout exige carrinho nao vazio
  [ok] confirmacao exige pedido gerado

Fluxos:
  [ok] compra completa — 11 passos
  [ok] pagamento recusado — 4 passos

Resultado: 0 erros | 2 avisos
  [aviso] produto -> carrinho via botao_adicionar: sem transição alternativa para produto.estoque igual a 0
  [aviso] carrinho -> checkout via botao_finalizar: sem transição alternativa para carrinho.quantidade igual a 0



In [13]:
# Inspecionando partes específicas do modelo
code_inspect = '''
(require "webnavi.rkt")
(require "loja.rkt")

(display "=== Páginas e suas rotas ===\n")
(for ([p (wn-site-pages loja)])
  (displayln (list (wn-page-name p)
                   "->" (wn-page-route p)
                   "|" (length (wn-page-elements p)) "elementos")))

(display "\n=== Transições com guardas ===\n")
(for ([tr (wn-site-transitions loja)]
      #:when (wn-transition-guard tr))
  (displayln (list (wn-transition-from tr) "->" (wn-transition-to tr))))

(display "\n=== Fluxos declarados ===\n")
(for ([fl (wn-site-flows loja)])
  (displayln (list (wn-flow-name fl)
                   "|" (length (wn-flow-steps fl)) "passos")))
'''
print(run_racket(code_inspect))

=== Páginas e suas rotas ===
(inicio -> / | 2 elementos)
(login -> /login | 4 elementos)
(catalogo -> /produtos | 2 elementos)
(produto -> /produto | 2 elementos)
(carrinho -> /carrinho | 2 elementos)
(checkout -> /checkout | 5 elementos)
(confirmacao -> /confirmacao | 2 elementos)
(erro_pagamento -> /checkout/erro | 1 elementos)

=== Transições com guardas ===
(login -> catalogo)
(login -> login)
(produto -> carrinho)
(carrinho -> checkout)
(checkout -> confirmacao)
(checkout -> erro_pagamento)

=== Fluxos declarados ===
((compra completa) | 11 passos)
((pagamento recusado) | 4 passos)



---
## 7. Geração de Testes Cypress

A função `generate-cypress` percorre os passos de um fluxo e gera um arquivo de teste JavaScript para o framework Cypress. O gerador:

- Resolve referências de datasets (`usuario_valido.email` → `"user@loja.com"`)
- Usa os seletores declarados nos elementos (`seletor "#email"` → `cy.get('#email')`)
- Gera asserções de URL a partir das rotas (`aguardar pagina login` → `cy.url().should('include', '/login')`)
- Gera asserções de visibilidade (`aguardar aviso_erro visivel` → `cy.get(...).should('be.visible')`)

Para elementos sem seletor declarado, o gerador usa o atributo `data-webnavi` como convenção: `[data-webnavi="nome_elemento"]`.

In [14]:
# Gera o teste Cypress para o fluxo 'compra completa'
code_cypress_compra = '''
(require "webnavi.rkt")
(require "loja.rkt")
(display (generate-cypress loja (quote (compra completa))))
'''
print(run_racket(code_cypress_compra))

// GERADO AUTOMATICAMENTE de WebNavi Scheme :: fluxo compra completa
describe('fluxo: compra completa', () => {
  it('deve navegar de inicio até confirmacao', () => {
    cy.visit('/');
    cy.get('[data-webnavi="botao_login"]').click();
    cy.url().should('include', '/login');
    cy.get('#email').type('user@loja.com');
    cy.get('#senha').type('Senha@123');
    cy.get('#entrar').click();
    cy.url().should('include', '/produtos');
    cy.get('[data-webnavi="botao_buscar"]').click();
    cy.url().should('include', '/produto');
    cy.get('#adicionar').click();
    cy.url().should('include', '/carrinho');
    cy.get('#finalizar').click();
    cy.url().should('include', '/checkout');
    cy.get('#cartao').type('4111111111111111');
    cy.get('#cvv').type('123');
    cy.get('[data-webnavi="campo_validade"]').type('12/28');
    cy.get('#pagar').click();
    cy.url().should('include', '/confirmacao');
  });
});



In [15]:
# Gera o teste Cypress para o fluxo 'pagamento recusado'
code_cypress_recusado = '''
(require "webnavi.rkt")
(require "loja.rkt")
(display (generate-cypress loja (quote (pagamento recusado))))
'''
print(run_racket(code_cypress_recusado))

// GERADO AUTOMATICAMENTE de WebNavi Scheme :: fluxo pagamento recusado
describe('fluxo: pagamento recusado', () => {
  it('deve navegar de checkout até erro_pagamento', () => {
    cy.visit('/checkout');
    cy.get('#cartao').type('0000000000000000');
    cy.get('#cvv').type('000');
    cy.get('[data-webnavi="campo_validade"]').type('01/20');
    cy.get('#pagar').click();
    cy.url().should('include', '/checkout/erro');
  });
});



---
## 8. Conclusão

### O que a WebNavi faz

- Define fluxos de navegação web em português com sintaxe de s-expressions
- Valida estrutura e referências em tempo de expansão de macros
- Verifica propriedades do grafo (alcançabilidade, exclusividade de guardas, invariantes)
- Gera testes Cypress a partir de fluxos declarativos

### Por que é uma DSL em Scheme, não um parser externo

A versão anterior usava Python para parsear uma linguagem textual própria. A versão atual usa `define-syntax` / `syntax-rules` do Racket para criar **formas sintáticas nativas** — o compilador Racket é o parser, e as macros são a gramática. Isso elimina a dependência de um parser externo e demonstra o poder de **macros higiênicas** como mecanismo de extensão de linguagem.

### Limitações atuais

| Limitação | Descrição |
|---|---|
| Verificação de invariantes | Limitada a antecedentes do tipo `(estiver em página)` |
| Análise de guardas | Cobre padrão `maior X que 0` mas não completude lógica geral |
| Geração de testes | Apenas Cypress; Playwright não implementado |
| Sem execução real | A DSL modela e valida — não executa o browser |

### Próximos passos possíveis

1. Estender a verificação de invariantes para antecedentes arbitrários
2. Adicionar backend de geração para Playwright
3. Implementar simulação de fluxo com estado (execução simbólica)
4. Adicionar visualização do grafo de navegação (DOT/Graphviz)